# Bonus 04 — LiteLLM Gateway Pattern
**Optional | After Lab 1A and Lab 5 | Colab CPU | OpenAI API key**

Lab 1A: one client, swap `base_url`. Lab 5: *you* were the server. A **gateway** sits in the middle so product code never names a provider.

LiteLLM keeps OpenAI-shaped `messages` and a `model` string like `openai/gpt-4o-mini` or `groq/llama-3.1-8b-instant`. This notebook uses the **Python SDK** (Colab-friendly). The proxy server (`litellm --model ...`) is stretch on a laptop.

> LiteLLM moves fast. If a cell errors, read the traceback once, then check the [docs](https://docs.litellm.ai/).


In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} litellm openai pandas python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

Groq is optional. If a `GROQ_API_KEY` exists (Colab Secret or `.env`), LiteLLM picks it up from the environment and a `groq/` route appears in Part C.

In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # not on Colab, or no such secret
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("Groq route available:", bool(GROQ_API_KEY))

## A. Baseline — direct OpenAI client

This is what most apps start with: provider baked into the constructor.


In [ ]:
messages = [
    {"role": "system", "content": "You are a concise LLM deployment coach."},
    {"role": "user", "content": "When should a team add an LLM gateway? Three bullets."},
]
print(client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, temperature=0.2).choices[0].message.content)


## B. Same request through LiteLLM

The `openai/` prefix is the provider. Change the prefix, keep `messages`.


In [ ]:
from litellm import completion

r = completion(model=f"openai/{DEFAULT_MODEL}", messages=messages, temperature=0.2)
print(r.choices[0].message.content)


## C. Routing is a table, not an if-ladder

In production this table is YAML / env / a gateway UI. Product code calls `gateway_chat("fast", prompt)`.


In [ ]:
import time
import pandas as pd

ROUTES = {
    "fast": f"openai/{DEFAULT_MODEL}",
    "quality": "openai/gpt-4o",
}
if GROQ_API_KEY:
    ROUTES["groq"] = "groq/llama-3.1-8b-instant"

call_log = []

def gateway_chat(route: str, prompt: str) -> str:
    model = ROUTES[route]
    t0 = time.time()
    result = completion(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise LLM deployment coach."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    usage = getattr(result, "usage", None)
    call_log.append({
        "route": route,
        "model": model,
        "latency_ms": int((time.time() - t0) * 1000),
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
    })
    return result.choices[0].message.content

print(gateway_chat("fast", "Explain fallback routing in two sentences."))
print(pd.DataFrame(call_log))


## D. Streaming through the same function


In [ ]:
stream = completion(
    model=f"openai/{DEFAULT_MODEL}",
    messages=[{"role": "user", "content": "Explain an LLM gateway in four short sentences."}],
    stream=True,
)
n = 0
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    n += len(delta)
    print(delta, end="")
print("\n\ncharacters streamed:", n)


## E. What the proxy adds (read, do not run on Colab)

```
App  →  OpenAI(base_url="http://gateway:4000/v1")  →  LiteLLM proxy  →  OpenAI / Groq / vLLM / Ollama
```

On a laptop:

```bash
uv pip install "litellm[proxy]"
litellm --model openai/gpt-4o-mini --port 4000
```

That is Lab 5's idea as a product: one `base_url`, many backends, plus retries and budgets.

## Bonus 04 complete

- [ ] Direct OpenAI call
- [ ] Same call via `completion(model="openai/...")`
- [ ] Route table + latency log
- [ ] Streamed tokens

## Stretch

1. If `GROQ_API_KEY` exists, call `gateway_chat("groq", ...)` and compare latency.
2. Wrap `gateway_chat` in try/except: on failure, retry route `"fast"`.
3. Laptop: start the proxy and point `OpenAI(base_url="http://localhost:4000/v1")`.

Next: [Bonus 06 — Ollama](06_ollama_local.md) for a local backend, or [Lab 5](../05_Serving_API/README.md).
